# 01 — Variable-only events: Strict vs Variable 1nt

**Question:** When we relax the splice site boundary by 1nt, which splicing events does SUPPA newly detect that strict mode missed?

**Why this matters:** Strict mode requires exact splice site matches; variable 1nt tolerates a 1 nucleotide shift. The relaxation reveals more events. We need to know exactly which events are new, because the next step (notebook 03) compares PSI values event-by-event between the two modes, and that only works if we can match the same event across both.

**Data:** IOE catalogues from generateEvents (no expression yet). Event types: A3, A5, RI.

**The obstacle:** the two modes label the same event with different coordinate formats, so we cannot compare IDs directly until we translate one into the other.

| Event | Strict ID format | Variable ID format |
|-------|-----------------|-------------------|
| RI | `gene;RI:chr:s1:e1-s2:e2:strand` | `gene;RI:chr:e1-s2:strand` |
| A3 | `gene;A3:chr:e1-s2:e1-s3:strand` | `gene;A3:chr:s2:s3:strand` |
| A5 | `gene;A5:chr:e2-s3:e1-s3:strand` | `gene;A5:chr:e2:e1:strand` |

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# --- Paths ---
IOE_STRICT = "/Users/gricey/Desktop/Internship/data/output_strict/events"
IOE_VAR1   = "/Users/gricey/Desktop/Internship/data/output_variable_1nt/events"
OUTPUT_DIR = "/Users/gricey/Desktop/Internship/data/ioe_diff"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EVENTS       = ["A3", "A5", "RI"]
SUFFIX_STRICT = "strict"
SUFFIX_VAR1   = "variable_1"

# --- Table helpers ---

def style_table(df, caption=""):
    styled = df.style\
        .set_properties(**{
            "font-size": "12px",
            "font-weight": "bold",
            "border": "1px solid #ddd",
            "padding": "6px 12px",
            "text-align": "center"
        })\
        .set_table_styles([
            {"selector": "th", "props": [
                ("background-color", "#2196F3"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("padding", "6px 12px"),
                ("text-align", "center")
            ]},
            {"selector": "tr:nth-child(even)", "props": [
                ("background-color", "#f2f2f2")
            ]},
        ])\
        .hide(axis="index")
    if caption:
        styled = styled.set_caption(caption)
    return styled


def export_table_png(df, filepath, title="", figsize=None):
    if figsize is None:
        figsize = (len(df.columns) * 2.2, len(df) * 0.6 + 0.8)
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.6)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#2196F3")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#f2f2f2")
        else:
            cell.set_facecolor("white")
        cell.set_edgecolor("#dddddd")
    if title:
        ax.set_title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved -> {filepath}")

print("Setup done!")

Setup done!


## The translation step: reformat strict IDs into variable format

**Why:** variable mode drops some coordinates that strict keeps. To match an event across both modes, we rewrite each strict ID keeping only the coordinates variable uses.

**How (strand matters):** A3 and A5 relax a biologically defined boundary (the 3' or 5' splice site). On the minus strand that boundary sits on the opposite genomic side, so the coordinates we keep flip:
- **RI**: keep the inner `e1-s2` pair (same for both strands)
- **A3**: plus strand keep right sides `s2:s3`, minus strand keep left sides `e2:e1`
- **A5**: plus strand keep left sides `e2:e1`, minus strand keep right sides `s2:s3`

The sanity check proves the translation is correct on known examples before we trust it on thousands of rows. If it fails, everything downstream is wrong.

In [2]:
def reformat_strict_event_id(event_id, event_type):
    try:
        gene_part, rest = event_id.split(";")
        tokens = rest.split(":")
        etype  = tokens[0]
        chrom  = tokens[1]
        strand = tokens[-1]
        coords = tokens[2:-1]

        if event_type == "RI":
            # same for both strands: keep the hyphenated inner pair e1-s2
            inner = [c for c in coords if "-" in c]
            if len(inner) != 1:
                return None
            return f"{gene_part};{etype}:{chrom}:{inner[0]}:{strand}"

        elif event_type == "A3":
            if len(coords) != 2:
                return None
            if strand == "+":
                # plus: e1-s2 | e1-s3 -> keep right sides: s2:s3
                c1 = coords[0].split("-")[1]
                c2 = coords[1].split("-")[1]
            else:
                # minus: e2-s3 | e1-s3 -> keep left sides: e2:e1
                c1 = coords[0].split("-")[0]
                c2 = coords[1].split("-")[0]
            return f"{gene_part};{etype}:{chrom}:{c1}:{c2}:{strand}"

        elif event_type == "A5":
            if len(coords) != 2:
                return None
            if strand == "+":
                # plus: e2-s3 | e1-s3 -> keep left sides: e2:e1
                c1 = coords[0].split("-")[0]
                c2 = coords[1].split("-")[0]
            else:
                # minus: e1-s2 | e1-s3 -> keep right sides: s2:s3
                c1 = coords[0].split("-")[1]
                c2 = coords[1].split("-")[1]
            return f"{gene_part};{etype}:{chrom}:{c1}:{c2}:{strand}"

    except Exception:
        return None


# Sanity checks using real examples from terminal
tests = [
    ("RI", "ENSMUSG00000019082;RI:7:141011934:141012052-141012527:141012666:-",
            "ENSMUSG00000019082;RI:7:141012052-141012527:-"),
    ("A3", "ENSMUSG00000057363;A3:1:43829804-43836405:43829710-43836405:-",
            "ENSMUSG00000057363;A3:1:43829804:43829710:-"),
    ("A5", "ENSMUSG00000000088;A5:9:57437645-57438952:57436355-57438952:+",
            "ENSMUSG00000000088;A5:9:57437645:57436355:+"),
]

print("Sanity checks:")
all_ok = True
for etype, strict_id, expected in tests:
    result = reformat_strict_event_id(strict_id, etype)
    ok = result == expected
    if not ok:
        all_ok = False
    print(f"  {etype}: {'OK' if ok else f'MISMATCH -> got {result}'}")

print()
print("All sanity checks passed!" if all_ok else "Some checks failed, review reformatting logic.")

Sanity checks:
  RI: OK
  A3: OK
  A5: OK

All sanity checks passed!


## Step 1 — Load the catalogues

Read both IOE files per event type. Each row is one event. The printed counts show the raw gap between modes, which is what we are about to explain.

In [3]:
ioe_data = {}

for event in EVENTS:
    ioe_data[event] = {
        "strict": pd.read_csv(f"{IOE_STRICT}/events_{event}_{SUFFIX_STRICT}.ioe", sep="\t"),
        "var1":   pd.read_csv(f"{IOE_VAR1}/events_{event}_{SUFFIX_VAR1}.ioe",   sep="\t"),
    }
    s = len(ioe_data[event]["strict"])
    v = len(ioe_data[event]["var1"])
    print(f"{event}  |  n_e_strict = {s:>6}  |  n_e_var1 = {v:>6}  |  difference = {v-s:>6}")

A3  |  n_e_strict =  11163  |  n_e_var1 =  22931  |  difference =  11768
A5  |  n_e_strict =   9629  |  n_e_var1 =  19455  |  difference =   9826
RI  |  n_e_strict =   5315  |  n_e_var1 =  17336  |  difference =  12021


## Step 2 — Classify every variable event

Reformat all strict IDs, then check each variable event: does its ID appear in the reformatted strict set?

- **n_e_shared** — yes, detected in both modes (these can be compared in notebook 03)
- **n_e_var_only** — no, only in variable, known Ensembl gene
- **n_e_var_novel** — no, only in variable, no Ensembl ID (coordinate-based)

Every variable event lands in exactly one bucket, so the three always sum to n_e_var1.

**Why event level:** PSI is a property of a single event, so ΔPSI in notebook 03 needs event-by-event pairing. Gene level cannot provide that.

In [4]:
populations = {}  # stores event ID sets and dataframes per event type
rows = []

for event in EVENTS:
    strict_df = ioe_data[event]["strict"].copy()
    var1_df   = ioe_data[event]["var1"]

    # All variable event IDs
    var1_ids = set(var1_df["event_id"])

    # Reformat strict IDs to variable format
    strict_df["event_id_var_fmt"] = strict_df["event_id"].apply(
        lambda x: reformat_strict_event_id(x, event)
    )
    n_reformatted = strict_df["event_id_var_fmt"].notna().sum()
    print(f"{event}: {n_reformatted}/{len(strict_df)} strict events reformatted successfully")

    strict_var_fmt = set(strict_df["event_id_var_fmt"].dropna())

    # Classify variable events
    shared      = var1_ids & strict_var_fmt
    var_only    = var1_ids - strict_var_fmt
    var_only_e  = {e for e in var_only if e.split(";")[0].startswith("ENSMUSG")}
    var_only_n  = {e for e in var_only if not e.split(";")[0].startswith("ENSMUSG")}

    populations[event] = {
        "shared":          shared,
        "var_only":        var_only_e,
        "var_novel":       var_only_n,
        "strict_var_fmt":  strict_var_fmt,
        "strict_df":       strict_df,
        "var1_df":         var1_df,
    }

    rows.append({
        "Event":           event,
        "n_e_strict":      len(strict_df),
        "n_e_var1":        len(var1_df),
        "n_e_shared":      len(shared),
        "n_e_var_only":    len(var_only_e),
        "n_e_var_novel":   len(var_only_n),
    })

print()
df_pop = pd.DataFrame(rows)
display(style_table(df_pop, "Table — Event-level comparison: Strict vs Variable 1nt"))
export_table_png(
    df_pop,
    "../../figures/plots/table_event_populations.png",
    title="Event-level comparison: Strict vs Variable 1nt",
    figsize=(14, 3)
)

A3: 11163/11163 strict events reformatted successfully
A5: 9629/9629 strict events reformatted successfully
RI: 5315/5315 strict events reformatted successfully



Event,n_e_strict,n_e_var1,n_e_shared,n_e_var_only,n_e_var_novel
A3,11163,22931,10750,12181,0
A5,9629,19455,9251,10204,0
RI,5315,17336,5073,12262,1


Saved -> ../../figures/plots/table_event_populations.png


## Gene-level view: which genes are new in variable?

This answers the original biological question separately, at gene level (`n_g_`): which genes had zero events in strict but appear in variable? These are genes the strict boundary was completely blind to. This is the view used earlier to pick IGV candidate genes.

In [5]:
# --- Gene-level view: which genes are newly detected in variable? ---
# This answers the original question: relaxing the boundary by 1nt reveals
# events in genes that strict detected zero events for.

rows_gene = []

for event in EVENTS:
    strict_df = ioe_data[event]["strict"]
    var1_df   = ioe_data[event]["var1"]

    # Gene sets (ENSMUSG only)
    strict_genes = set(g for g in strict_df["gene_id"] if str(g).startswith("ENSMUSG"))
    var1_genes   = set(g for g in var1_df["gene_id"]   if str(g).startswith("ENSMUSG"))

    # Genes only in variable (never detected in strict)
    new_genes = var1_genes - strict_genes

    rows_gene.append({
        "Event":          event,
        "n_g_strict":     len(strict_genes),
        "n_g_var1":       len(var1_genes),
        "n_g_new (v-s)":        len(new_genes),
    })

df_gene = pd.DataFrame(rows_gene)
display(style_table(df_gene, "Gene-level: genes newly detected in Variable 1nt"))
export_table_png(
    df_gene,
    "../../figures/plots/table_gene_new.png",
    title="Genes newly detected in Variable 1nt",
    figsize=(11, 3)
)

Event,n_g_strict,n_g_var1,n_g_new (v-s)
A3,6684,10107,3423
A5,5905,9002,3097
RI,3163,7567,4404


Saved -> ../../figures/plots/table_gene_new.png


## Step 3 — Export variable-only events

Save the variable-only events to IOE files and gene lists. These are the concrete deliverables: the material strict missed, ready for IGV and for the PSI comparison.

In [6]:
for event in EVENTS:
    pop     = populations[event]
    var1_df = pop["var1_df"]

    # All variable-only events (ENSMUSG + novel)
    var_only_all = pop["var_only"] | pop["var_novel"]

    # Filter variable IOE to variable-only events
    ioe_unique = var1_df[var1_df["event_id"].isin(var_only_all)]

    # Unique ENSMUSG gene IDs
    unique_genes = sorted(set(
        e.split(";")[0] for e in pop["var_only"]
    ))

    # Save files
    ioe_unique.to_csv(f"{OUTPUT_DIR}/{event}_var1_unique.ioe", sep="\t", index=False)
    with open(f"{OUTPUT_DIR}/{event}_var1_unique_genes.txt", "w") as f:
        for gene in unique_genes:
            f.write(gene + "\n")

    print(f"{event}:")
    print(f"  n_e_var_only = {len(pop['var_only'])} events ({len(unique_genes)} unique genes)")
    print(f"  n_e_var_novel = {len(pop['var_novel'])} events (no Ensembl ID)")
    print()

A3:
  n_e_var_only = 12181 events (6195 unique genes)
  n_e_var_novel = 0 events (no Ensembl ID)

A5:
  n_e_var_only = 10204 events (5312 unique genes)
  n_e_var_novel = 0 events (no Ensembl ID)

RI:
  n_e_var_only = 12262 events (6045 unique genes)
  n_e_var_novel = 1 events (no Ensembl ID)



## Step 4 — Superset check (validation)

If the translation is correct, every strict event must be found in variable (strict is a subset of variable). So "strict events missing from variable" must be zero for all event types. This is the proof that the matching is sound — the result we show Khushi to validate the method.

In [7]:
rows_check = []

for event in EVENTS:
    pop    = populations[event]
    var1_ids           = set(pop["var1_df"]["event_id"])
    strict_var_fmt     = pop["strict_var_fmt"]
    n_strict_missing   = len(strict_var_fmt - var1_ids)
    n_total_accounted  = len(pop["shared"]) + len(pop["var_only"]) + len(pop["var_novel"])
    n_var_total        = len(pop["var1_df"])

    rows_check.append({
        "Event":                    event,
        "n_e_var1 (total)": n_var_total,
        "n_e_shared + var_only + novel": n_total_accounted,
        "accounting OK?": "yes" if n_total_accounted == n_var_total else "MISMATCH",
        "strict events missing from var": n_strict_missing,
        "superset OK?": "yes" if n_strict_missing == 0 else f"{n_strict_missing} missing!",
    })

df_check = pd.DataFrame(rows_check)
display(style_table(df_check, "Superset check: variable contains all strict events"))
export_table_png(
    df_check,
    "../../figures/plots/table_superset_check.png",
    title="Superset check: variable contains all strict events",
    figsize=(18, 3)
)

Event,n_e_var1 (total),n_e_shared + var_only + novel,accounting OK?,strict events missing from var,superset OK?
A3,22931,22931,yes,0,yes
A5,19455,19455,yes,0,yes
RI,17336,17336,yes,0,yes


Saved -> ../../figures/plots/table_superset_check.png
